# STAC AI Workshop · NYC · 21 October 2026

## 02 · Relevant, applicable and sufficient evidence

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Distinguish retrieved relevance from time eligibility
2. State who defines claim-specific evidence requirements
3. Trace impact through a branching graph
4. Test valid and invalid repairs without changing unrelated claims

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

In [1]:
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

Offline workbook ready: no API key, Node.js, GPU or model download.


## What RAG does
**Retrieve → add retrieved passages to model context → generate an answer.** A retrieval service can find a relevant document without establishing that it was available, applicable or sufficient for the claim.

This offline exercise tests evidence selection and dependencies; it does not call a search service or LLM. Policy owners and domain experts supply the requirements.

## 1 · Baseline / Future amendment / Missing covenant
The cutoff is 08:55 New York on 20 October 2026. Publication **and** availability must precede the cutoff. The signed covenant is required for the current ratio claim. Comfortable headroom also requires scenario assessment; that remains unresolved.

In [2]:
rag_results={case:rag_case(case) for case in ("baseline","future","missing")}
show_table([{"Case":k,"Current ratio claim":v["current_claim"],"Comfort claim":v["comfort_claim"],"Gate":v["gate"]} for k,v in rag_results.items()])
show_table([{"Source":s["id"],"Title":s["title"],"Published":s["published"],"Available":s["available"],"In context":s["inContext"],"Time eligible":s["eligible"]} for s in rag_results["future"]["sources"]])
assert rag_results["baseline"]["bound"] and not rag_results["missing"]["bound"]
assert any(s["inContext"] and not s["eligible"] for s in rag_results["future"]["sources"])
for ref in load_fixture("rag-reference.json"):
    got=rag_results[ref["id"]]
    assert got["bound"]==ref["bound"]
    assert got["current_claim"]==ref["requirements"][0]["status"]
    for a,b in zip(got["sources"],ref["sources"]):
        assert (a["eligible"],a["inContext"])==(b["eligible"],b["inContext"])
print("PASS: source eligibility and claim support match the web cases.")

Case,Current ratio claim,Comfort claim,Gate
baseline,SUPPORTED IN FIXTURE,UNRESOLVED,HOLD
future,SUPPORTED IN FIXTURE,UNRESOLVED,HOLD
missing,UNRESOLVED,UNRESOLVED,HOLD


Source,Title,Published,Available,In context,Time eligible
E1,Harbor Metals filing,2026-10-20T12:00:00Z,2026-10-20T12:02:00Z,True,True
E2,Signed covenant,2026-10-20T12:08:00Z,2026-10-20T12:10:00Z,True,True
E3,Operations update,2026-10-20T12:40:00Z,2026-10-20T12:42:00Z,True,True
E4,Later covenant amendment,2026-10-20T13:05:00Z,2026-10-20T13:07:00Z,True,False


PASS: source eligibility and claim support match the web cases.


## 2 · Evidence impact and repair
The web extension switches to an **EPS evidence graph**, not the covenant calculation. Source definitions are explicit:

Actual EPS + consensus → surprise → positive signal → recommendation.
Actual EPS also supports the reported actual and earnings-risk branch. An independent review supports a separate OR branch. Market-open status is unrelated.

Inspect which branches can be affected structurally versus which actually become blocked.

In [3]:
fixture=load_fixture("repair-fixtures.json")
graph=fixture["graph"]
case=fixture["cases"][0]  # missing_actual
before=evaluate(graph,case["records"])
diagnosis=diagnose(graph,before)
show_table([{"Node":n,"Type":graph[n]["type"],"State":s["state"],"Value":s["value"],"Reason":s.get("reason","")} for n,s in before.items()])
show_table([{"Requested node":r["node"],"Request":r["action"],"Requirements":str(r["constraints"])} for r in diagnosis["requests"]])
print("Potentially affected:",", ".join(diagnosis["structural_potentially_affected"]))
print("Actually blocked:",", ".join(diagnosis["actually_blocked"]))
assert "review_supported" in diagnosis["structural_potentially_affected"]
assert before["review_supported"]["state"]=="ok"

Node,Type,State,Value,Reason
actual,source,blocked,None,no_eligible_source
consensus,source,ok,2,
surprise,compute,blocked,None,dependency_failed
positive,compute,blocked,None,dependency_failed
recommendation,compute,blocked,None,dependency_failed
report_actual,compute,blocked,None,dependency_failed
earnings_risk,compute,blocked,None,dependency_failed
independent_review,constant,ok,True,
review_supported,or,ok,True,
unrelated,constant,ok,market-open,


Requested node,Request,Requirements
actual,obtain_eligible_evidence,"{'issuer': 'Harbor', 'period': 'Q2', 'metric': 'actual', 'eligible': True, 'operative': True}"


Potentially affected: earnings_risk, positive, recommendation, report_actual, review_supported, surprise
Actually blocked: actual, surprise, positive, recommendation, report_actual, earnings_risk


## 3 · Apply an invalid repair, then the supplied valid one
Wrong issuer, stale evidence and unauthorized corrections must not resolve the recommendation. Repair requests are generated by the graph; the replacement evidence is supplied by the synthetic fixture.

In [4]:
invalid,_=apply_repair(case["records"],case["controls"][0]["repair"])
invalid_state=evaluate(graph,invalid)
assert invalid_state["recommendation"]["state"]=="blocked"
repaired,audit=apply_repair(case["records"],case["repair"])
after=evaluate(graph,repaired)
show_table([{"Node":n,"Before":before[n]["state"],"After":after[n]["state"],"Resolved value":after[n]["value"]} for n in graph])
assert all(v["state"]=="ok" for v in after.values())
assert before["unrelated"]==after["unrelated"]
assert before["review_supported"]["value"]==after["review_supported"]["value"]

Node,Before,After,Resolved value
actual,blocked,ok,3
consensus,ok,ok,2
surprise,blocked,ok,1
positive,blocked,ok,True
recommendation,blocked,ok,buy
report_actual,blocked,ok,3
earnings_risk,blocked,ok,True
independent_review,ok,ok,True
review_supported,ok,ok,True
unrelated,ok,ok,market-open


## 4 · Verify all four valid and seventeen invalid repairs
The same generic evaluator runs each supplied case. It does not encode case names as outcomes.

In [5]:
summary=[]; invalid_count=0
references={r["id"]:r for r in load_fixture("repair-reference.json")}
for case in fixture["cases"]:
    b=evaluate(graph,case["records"]); d=diagnose(graph,b)
    repaired,_=apply_repair(case["records"],case["repair"]); a=evaluate(graph,repaired)
    assert b==references[case["id"]]["before"]
    assert d==references[case["id"]]["diagnosis"]
    assert a==references[case["id"]]["after"] and all(v["state"]=="ok" for v in a.values())
    assert b["unrelated"]==a["unrelated"]
    for control,reference in zip(case["controls"],references[case["id"]]["controls"]):
        bad,_=apply_repair(case["records"],control["repair"]); state=evaluate(graph,bad)
        assert state==reference["after"] and state["recommendation"]["state"]=="blocked"
        invalid_count+=1
    summary.append({"Case":case["id"],"Valid repair":"Resolved","Rejected controls":len(case["controls"]),"Unrelated branch":"Unchanged"})
show_table(summary)
assert invalid_count==17
print("PASS: 4 valid repairs and 17 invalid controls match browser outputs.")

Case,Valid repair,Rejected controls,Unrelated branch
missing_actual,Resolved,4,Unchanged
stale_consensus,Resolved,4,Unchanged
conflict_actual,Resolved,4,Unchanged
two_missing,Resolved,5,Unchanged


PASS: 4 valid repairs and 17 invalid controls match browser outputs.


## Learning check
Why can a structural descendant stay supported? **An independent valid route can still support an OR node.**

Who decides what is enough? **Domain experts and policy owners define claim-specific requirements.** The graph checks those requirements; it does not prove universal completeness or factual truth.

A URL is not a historical snapshot. Section 4 checks version availability and look-ahead leakage.

## Main takeaway
Adding a document is not the same as repairing the required dependency. The graph identifies the requirement, affected branches and whether a qualifying repair resolves them. This is synthetic evidence—not autonomous retrieval.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.